In [ ]:
!pip install -q -U smolagents "smolagents[litellm]" ddgs markdownify
# [litellm] lets smolagents communicate with Ollama through LiteLLM
# markdownify only needed for VisitWebpageTool in section containing the full Alfred agent. That tool downloads a web page and uses markdownify to convert the HTML to markdown so the agent gets readable text instead of raw HTML. If the package is missing, the tool fails with an error asking us to install it.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.9/36.9 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.3/278.3 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 9.7 MB/s eta 0:00:00


In [ ]:
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

Selecting previously unselected package zstd.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess, time, requests
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
  try:
    requests.get("http://localhost:11434", timeout=1)
    print("Ollama server is up")
    break
  except Exception:
    time.sleep(1)
else:
  print("Server did not start. Re-run this cell.")

Ollama server is up


In [ ]:
# We should ideally use Qwen2.5-Coder-32B but it is too big for a free T4. 14B is the closest size that fits;
# but if it is too slow or runs out of memory, switch to "qwen2.5-coder:7b" (less reliable).
MODEL_TAG = "qwen2.5-coder:14b"
!ollama pull {MODEL_TAG}

In [ ]:
BACKEND = "ollama" # or "hf"

from smolagents import LiteLLMModel, InferenceClientModel
from smolagents.models import ChatMessage

if BACKEND == "ollama":
  model = LiteLLMModel(
      model_id=f"ollama_chat/{MODEL_TAG}",
      api_base="http://localhost:11434",
      num_ctx=16384, #agent prompts grow each step; Ollama's default context is far too  small.
  )
else:
  from huggingface_hub import notebook_login
  notebook_login()
  model = InferenceClientModel()

# connection test
print(
    model.generate([
        ChatMessage(
            role="user",
            content=[{"type": "text", "text": "Say hello in one short sentence."}]
        )
    ]).content
)

Hello!


In [ ]:
def show_tokens(agent):
  # attribute may differ between smolagents versions
  try:
    print("Token usage:", agent.monitor.get_total_token_counts())
  except Exception as e:
    print("Token counter not available:", e)

In [ ]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, tool
search_tool = DuckDuckGoSearchTool(max_results=5)
try:
  print(search_tool("best party music")[:600])
except Exception as e:
  print("DuckDuckGoSearchTool failed:", e)
  # fallback with the same tool name (web_search), so agent prompts and generated code dont change
  from ddgs import DDGS

  @tool
  def web_search(query: str) -> str:
    """Search the web and return the top results as short text
    Args:
    query: the search query
    """
    try:
      with DDGS() as d:
        results = d.text(query, max_results=5)
    except Exception as e:
      return f"Search failed: {e}"
    return "\n".join(f"- {r['title']}: {r['body'][:200]} ({r['href']})" for r in results) or "No results found."

  search_tool = web_search


In [ ]:
agent = CodeAgent(tools=[search_tool], model=model, max_steps=6)
agent.run("Search for the best music recommendations for a party at the Wayne's mansion.")
show_tokens(agent)

In [ ]:
from smolagents import CodeAgent, tool

# Tool to suggest a menu based on the occasion
@tool
def suggest_menu(occasion: str) -> str:
  """
  Suggests a menu based on the occasion.
  Args:
  occasion (str): The type of occasion for the party. Allowed values are:
                  - "casual": Menu for casual party.
                  - "formal": Menu for formal party.
                  - "superhero": Menu for superhero party.
                  - "custom": Custom menu.
  """
  if occasion == "casual":
    return "Pizza, snacks and drinks."
  elif occasion == "formal":
    return "3-course dinner with wine and dessert."
  elif occasion == "superhero":
    return "Buffet with high-energy and healthy food."
  else:
    return "Custom menu for the butler."

  # Alfred, the butler, preparing the menu for the party
  agent = CodeAgent(tools=[suggest_menu], model=model, max_steps=5)

  # Preparing the menu for the party
  agent.run("Prepare a formal menu for the party.")
  show_tokens(agent)
